In [ ]:
import re
import os
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

import nltk
from nltk.stem import WordNetLemmatizer

# ================== INITIAL NLTK SETUP ==================

# Download WordNet (first run will download; later runs will just reuse)
#nltk.download("wordnet", quiet=True)
lemmatizer = WordNetLemmatizer()

# ================== SETTINGS ==================


START_URL = "https://www.bbc.com/"

MAX_PAGES = 20      # how many pages you need
MIN_WORDS = 500     # minimum words for each page
OUTPUT_DIR = "wordcloud_output"

# Base stopwords from wordcloud + custom colors
custom_stopwords = set(STOPWORDS)
custom_stopwords.update([
    # colors
    "red", "blue", "green", "yellow", "purple", "orange",
    "brown", "white", "black", "gray", "grey", "pink",
    "silver", "gold", "beige", "cyan", "magenta",

    # time words from news sites
    "ago", "hr", "hrs", "hour", "hours",
    "min", "mins", "minute", "minutes",
    "day", "days", "today", "yesterday",

    # site-generic words
    "bbc", "news", "live", "update"
]
)

# ================== HTML → CLEAN TEXT ==================

def get_clean_text(html: str) -> str:
    """
    Remove scripts/styles and return clean visible text.
    """
    soup = BeautifulSoup(html, "html.parser")

    # remove things that are not real content
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # get text and normalize spaces
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ================== LINK NORMALIZATION ==================

def normalize_link(href: str, base_url: str, base_netloc: str):
    """
    Turn relative links into absolute, keep only internal http/https links.
    """
    if not href:
        return None
    href = href.strip()

    # skip anchors, emails, javascript, etc.
    if href.startswith("#") or href.startswith("mailto:") or href.startswith("javascript:"):
        return None

    abs_url = urljoin(base_url, href)
    parsed = urlparse(abs_url)

    if parsed.scheme not in ("http", "https"):
        return None

    # only keep links inside the same website (same domain)
    if parsed.netloc != base_netloc:
        return None

    # drop #fragment at the end
    cleaned = parsed._replace(fragment="").geturl()
    return cleaned

# ================== TEXT NORMALIZATION (STOPWORDS + LEMMA) ==================

def normalize_text(text: str) -> str:
    """
    - Lowercase
    - Keep only alphabetic tokens
    - Remove stopwords (including colors)
    - Lemmatize (so animal/animals → animal)
    """
    # extract only alphabetic words
    words = re.findall(r"[A-Za-z]+", text.lower())
    cleaned = []

    for w in words:
        if w in custom_stopwords:
            continue

        # remove very short words
        if len(w) <= 2:
            continue

        # lemmatize as noun by default; this already merges plural to singular
        lemma = lemmatizer.lemmatize(w)

        if lemma in custom_stopwords:
            continue

        cleaned.append(lemma)

    # WordCloud will count these and merge same lemmas
    return " ".join(cleaned)

# ================== CRAWLER ==================

def crawl_site(start_url: str,
               max_pages: int = MAX_PAGES,
               min_words: int = MIN_WORDS):
    """
    BFS crawl starting from start_url and return a list of (url, text)
    where text length >= min_words.
    """
    parsed_start = urlparse(start_url)
    base_netloc = parsed_start.netloc

    to_visit = [start_url]
    visited = set()
    pages = []

    session = requests.Session()
    headers = {"User-Agent": "StudentCrawler/1.0 (for wordcloud assignment)"}

    while to_visit and len(pages) < max_pages:
        url = to_visit.pop(0)

        if url in visited:
            continue
        visited.add(url)

        try:
            resp = session.get(url, headers=headers, timeout=10)
            resp.raise_for_status()
        except Exception as e:
            print(f"[WARN] Could not fetch {url}: {e}")
            continue

        text = get_clean_text(resp.text)
        word_count = len(text.split())
        print(f"[INFO] Fetched {url} ({word_count} words)")

        # keep only pages with enough content
        if word_count >= min_words:
            pages.append((url, text))

        # collect new internal links to visit
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.find_all("a", href=True):
            norm = normalize_link(a["href"], url, base_netloc)
            if norm and norm not in visited and norm not in to_visit:
                to_visit.append(norm)

    return pages

# ================== WORD CLOUD GENERATION ==================

def generate_wordcloud(text: str, title: str, filename: str):
    """
    Make and save a word cloud image from cleaned text.
    """
    clean = normalize_text(text)

    wc = WordCloud(
        width=1600,
        height=900,
        background_color="white",
        max_words=200,
        stopwords=custom_stopwords
    ).generate(clean)

    plt.figure(figsize=(12, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

# ================== MAIN SCRIPT ==================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    pages = crawl_site(START_URL, MAX_PAGES, MIN_WORDS)
    print(f"\n[INFO] Collected {len(pages)} pages with at least {MIN_WORDS} words.\n")

    for idx, (url, text) in enumerate(pages, start=1):
        safe_idx = f"{idx:02d}"
        img_path = os.path.join(OUTPUT_DIR, f"page_{safe_idx}_wordcloud.png")
        txt_path = os.path.join(OUTPUT_DIR, f"page_{safe_idx}_text.txt")

        # save raw original text (for checking / report)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(f"URL: {url}\n\n")
            f.write(text)

        title = f"Word Cloud #{safe_idx}"
        generate_wordcloud(text, title, img_path)

        print(f"[OK] Saved {img_path} and {txt_path} for:\n     {url}\n")

    print("Done. Check the 'wordcloud_output' folder.")

if __name__ == "__main__":
    main()


[INFO] Fetched https://www.bbc.com/ (2052 words)
[INFO] Fetched https://www.bbc.com/news (1412 words)
[INFO] Fetched https://www.bbc.com/sport (1934 words)
[INFO] Fetched https://www.bbc.com/business (1549 words)
[INFO] Fetched https://www.bbc.com/innovation (1885 words)
[INFO] Fetched https://www.bbc.com/culture (1460 words)
[INFO] Fetched https://www.bbc.com/arts (1055 words)
[INFO] Fetched https://www.bbc.com/travel (2043 words)
[INFO] Fetched https://www.bbc.com/future-planet (1206 words)
[INFO] Fetched https://www.bbc.com/audio (732 words)
[INFO] Fetched https://www.bbc.com/video (1875 words)
[INFO] Fetched https://www.bbc.com/live (933 words)
[INFO] Fetched https://www.bbc.com/home (2028 words)
[INFO] Fetched https://www.bbc.com/news/topics/c2vdnvdg6xxt (1049 words)
[INFO] Fetched https://www.bbc.com/news/war-in-ukraine (1017 words)
[INFO] Fetched https://www.bbc.com/news/us-canada (1165 words)
[INFO] Fetched https://www.bbc.com/news/uk (1202 words)
[INFO] Fetched https://www.bbc